In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_question_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_question_window_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_sequences_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_window_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_window_sequences_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/train_valid_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/train_valid_sequences_quelevel.csv
/kaggle/input/datasets/nht123456789/simplektc-f1234-pykt-scripts/DKT_A_f0_preds.parquet
/kaggle/input/datasets/nht123456789/simplektc-f1234-pykt-scripts/fix_pykt_raw.py
/kaggle/input/datasets/nht123456789/simplektc-f1234-pykt-scrip

In [2]:
%%bash
# Ô Kaggle simpleKT-PQC fold 1–4 (bản 07/11 = ô 02/11, bỏ qrun_check vì đã chạy ở fold 0).
# Hai phiên song song trên hai tài khoản: FOLDS="1 2" và FOLDS="3 4" (~5.1 giờ mỗi phiên).
set -eo pipefail
cd /kaggle/working
[ -d pykt-toolkit ] || git clone -q https://github.com/pykt-team/pykt-toolkit.git
cd pykt-toolkit && git checkout -q eec087e
X=/kaggle/input/datasets/nhatanhadr/xes3g5m-extended
T=/kaggle/input/datasets/nht123456789/simplektc-f1234-pykt-scripts  # SỬA: Dataset script 02/11
REF=$T/DKT_A_f0_preds.parquet                    # mốc của điều kiện (3)
W=/kaggle/working
# C: giữ bản gốc (cho kiểm thử 1), vá simplekt.py, dừng nếu kiểm thử không đạt cả 5
P=pykt/models
[ -f $P/simplekt_orig.py ] || cp $P/simplekt.py $P/simplekt_orig.py
grep -q "PATCHED: simplekt_qmask" $P/simplekt.py || \
  python $T/simplekt_qmask_patch.py $P/simplekt.py
pip install -q -e . && pip install -q einops entmax
python $T/test_simplekt_qmask.py | tee $W/test_qmask.log
grep -q "ĐẠT cả 5" $W/test_qmask.log
mkdir -p data/peiyou
for f in train_valid_sequences test_question_window_sequences; do
  ln -sf $X/$f.csv data/peiyou/$f.csv
done
cp $T/*.py examples/ && cd examples && mkdir -p logs
M=simplekt; S=42; FOLDS="3 4"                    # SỬA: "1 2" hoặc "3 4"
trap 'kill $RAM 2>/dev/null || true' EXIT
for F in $FOLDS; do
  L=logs/${M}C_f${F}_s${S}
  ( while true; do echo "$(date +%T) $(free -g | awk '/Mem/ {print $3"/"$2}') GB"
      sleep 15; done ) > ${L}_ram.log &
  RAM=$!
  # C: emb_type qid_qmask và thư mục lưu riêng
  python -u wandb_${M}_train.py --dataset_name=peiyou --fold=$F --seed=$S \
    --emb_type=qid_qmask --use_wandb=0 --add_uuid=0 \
    --save_dir=sm_${M}C_f${F} > ${L}_train.log 2>&1
  CKPT=$(dirname $(find sm_${M}C_f${F} -name config.json | head -n 1))
  grep -q '"emb_type": "qid_qmask"' $CKPT/config.json
  T0=$SECONDS
  python -u predict_sharded.py --save_dir=$CKPT --n_shards=8 --bz=64 \
    > ${L}_predict.log 2>&1
  echo "predict_s=$((SECONDS-T0))" >> ${L}_predict.log
  kill $RAM
  # C: --variant=C; commit ghi rõ đã vá
  python pykt_to_registry.py --save_dir=$CKPT --train_log=${L}_train.log \
    --test_csv=$X/test.csv --variant=C --pykt_commit=eec087e+qmask \
    --runs_root=$W/runs --ref_preds=$REF --logs ${L}_predict.log ${L}_ram.log
  tail -3 ${L}_predict.log
  echo "RAM cao nhất fold $F: $(awk '{print $2}' ${L}_ram.log | sort -n | tail -1) GB"
done
cat $W/runs/registry.csv

đã vá pykt/models/simplekt.py: 11 chỗ sửa và hàm question_mask
model_name: simplekt, emb_type: qid
model_name: simplekt, emb_type: qid
model_name: simplekt, emb_type: qid_qmask
ĐẠT  1. emb_type qid giống pyKT gốc: chênh lớn nhất 0.00e+00
ĐẠT  2. câu 1 KC: qid_qmask = qid: chênh lớn nhất 0.00e+00
ĐẠT  3. không rò rỉ trong câu (qid_qmask): KC cùng câu 0.00e+00, câu kế tiếp ít nhất 5.17e-04
ĐẠT  4. đối chứng dương (qid rò rỉ): KC thứ 2 cùng câu đổi 3.45e-03
model_name: simplekt, emb_type: qid_qmask
ĐẠT  5. overfit một batch (qid_qmask): loss 0.695 -> 0.0000
KẾT QUẢ: ĐẠT cả 5
SIMPLEKT_C_f3_s42_20261107-1725 {'auc': 0.8166, 'acc': 0.8269, 'rmse': 0.3517, 'brier': 0.1237, 'ece': 0.0077, 'auprc_wrong': 0.5643, 'f1_wrong': 0.4326}
windowauclate_mean = 0.816645
windowacclate_mean = 0.826935
predict_s=8180
RAM cao nhất fold 3: 6/31 GB
SIMPLEKT_C_f4_s42_20261107-2002 {'auc': 0.8169, 'acc': 0.827, 'rmse': 0.3517, 'brier': 0.1237, 'ece': 0.01, 'auprc_wrong': 0.5649, 'f1_wrong': 0.4456}
windowauclat

/kaggle/working/pykt-toolkit/pykt/models/gkt.py:329: SyntaxWarning: invalid escape sequence '\m'
  this gate is different with that of the DKVMN. We used the input matrix to build the erase and add gates, rather than $\mathbf{v}_{t}$ vector in the DKVMN.
/kaggle/working/pykt-toolkit/pykt/models/folibikt.py:592: SyntaxWarning: invalid escape sequence '\T'
  * `base` is the constant used for calculating $\Theta$
/kaggle/working/pykt-toolkit/pykt/models/stablekt.py:592: SyntaxWarning: invalid escape sequence '\T'
  * `base` is the constant used for calculating $\Theta$
/kaggle/working/pykt-toolkit/pykt/models/extrakt.py:574: SyntaxWarning: invalid escape sequence '\T'
  * `base` is the constant used for calculating $\Theta$
